### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [103]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [104]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [105]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [106]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [107]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [108]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [109]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [110]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [111]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [112]:
# tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [113]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [114]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [115]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [116]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [117]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [118]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [119]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 1911, 1825, 1828], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [120]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [121]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [122]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [123]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [124]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [125]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


### **1. Vectorizar documentos**

In [126]:
#agregamos una semilla para que los resultados sean reproducibles
np.random.seed(42)
# Elegimos los 5 números al azar de los 11314 documentos de train
idxs = np.random.choice(11314, size=5, replace=False)
print(idxs)

[7492 3546 5582 4793 3813]


Vamos a definir una función para ir seleccionando de a uno los casos de mayor similitud y explicar la interpretación del resultado

In [127]:


import pandas as pd

nombres = newsgroups_train.target_names

def mostrar_similares(idx, n_caso):
    sim_cos = cosine_similarity(X_train[idx], X_train)[0]
    mas_sim = np.argsort(sim_cos)[::-1][1:6]  # los 5 más similares, excluyendo el mismo documento

    print(f"Caso {n_caso}  |  índice {idx}  |  clase: {nombres[y_train[idx]]}")

    tabla = pd.DataFrame({
        'índice': mas_sim,
        'clase': [nombres[y_train[i]] for i in mas_sim],
        'similitud': sim_cos[mas_sim].round(3),
        'misma clase': ['✓' if y_train[i] == y_train[idx] else '✗' for i in mas_sim],
    })
    display(tabla.style.hide(axis='index'))



In [128]:
#primer documento de los 5 elegidos al azar
mostrar_similares(idxs[0], 1)

Caso 1  |  índice 7492  |  clase: comp.sys.mac.hardware


índice,clase,similitud,misma clase
10935,comp.sys.mac.hardware,0.667000,✓
7258,comp.sys.ibm.pc.hardware,0.348000,✗
4971,comp.sys.mac.hardware,0.180000,✓
4303,misc.forsale,0.155000,✗
645,comp.sys.mac.hardware,0.141000,✓


En este primer caso se observa que en los 2 casos donde no coincide la clase, en el caso del índice 7258 es muy similar solo que se diferencia en ser IBM (7258) en lugar de mac como dice la clase del que se compara pero seguramente las palabras utilizadas en los documentos tienen bastante similitud.

In [129]:
# para el caso del índice 4303 vamos a ver qué palabras comparte con el documento a comparar.

def terminos_compartidos(i, j, n=10):
    a = X_train[i].toarray()[0]
    b = X_train[j].toarray()[0]
    aporte = a * b                      # cuánto aporta cada término a la similitud
    top = aporte.argsort()[::-1][:n]
    return [(idx2word[t], round(aporte[t], 3)) for t in top if aporte[t] > 0]

terminos_compartidos(7492, 4303)

[('cwru', np.float64(0.061)),
 ('po', np.float64(0.049)),
 ('edu', np.float64(0.013)),
 ('please', np.float64(0.012)),
 ('thanks', np.float64(0.012)),
 ('on', np.float64(0.003)),
 ('for', np.float64(0.003)),
 ('to', np.float64(0.002))]

Vemos que las palabras de mayor peso están relacionadas con el e-mail.

In [130]:
# segundo documento de los 5 elegidos al azar
mostrar_similares(idxs[1], 2)

Caso 2  |  índice 3546  |  clase: comp.os.ms-windows.misc


índice,clase,similitud,misma clase
5665,comp.sys.ibm.pc.hardware,0.204000,✗
2011,comp.sys.ibm.pc.hardware,0.192000,✗
8643,comp.sys.ibm.pc.hardware,0.172000,✗
1546,comp.sys.ibm.pc.hardware,0.171000,✗
8765,comp.sys.ibm.pc.hardware,0.162000,✗


Para este segundo caso vemos que no hay coincidencia en ninguno de los casos. Vemos que el documento de referencia  habla 
de componente de software y son misceláneas y en todos los casos de similitud también habla de computación, igualmente
se observa que tienen baja similitud.

In [131]:
#vemos el documento de referencia
print(newsgroups_train.data[3546])





     Don't bother if you have CPBackup or Fastback.  They all offer options 
not available in the stripped-down MS version (FROM CPS!).  Examples - no 
proprietary format (to save space), probably no direct DMA access, and no 
tape drive!


In [132]:
terminos_compartidos(3546, 5665)

[('dma', np.float64(0.178)),
 ('no', np.float64(0.005)),
 ('don', np.float64(0.004)),
 ('if', np.float64(0.003)),
 ('the', np.float64(0.003)),
 ('they', np.float64(0.003)),
 ('to', np.float64(0.002)),
 ('in', np.float64(0.002)),
 ('have', np.float64(0.002)),
 ('or', np.float64(0.001))]

Como se observa hay una sola palabra que comparte que habla de computadoras (software/hardware), pero evidentemente es la que mayor aporte tiene ya que es un término técnico y el resto  de las coincidencias igual que el caso anterior muchas stopwords no dan ningún tipo de contexto. Se sobreentiende que los otros casos de menor similitud tendrán el mismo patrón que el de mayor similitud.

In [133]:
# tercer documento de los 5 elegidos al azar
mostrar_similares(idxs[2], 3)

Caso 3  |  índice 5582  |  clase: misc.forsale


índice,clase,similitud,misma clase
5510,misc.forsale,0.462000,✓
4922,misc.forsale,0.300000,✓
4347,comp.graphics,0.274000,✗
8057,misc.forsale,0.208000,✓
4028,comp.graphics,0.169000,✗


Similar al caso del documento 2 en este al hablar de misceláneas vamos a ver el documento original a qué hace referencia

In [134]:
#vemos el documento de referencia
print(newsgroups_train.data[5582])

5.25" Internal Low density disk drive.

Monochrome monitor

8088 motherboard, built in parallel and serial ports, built in mono and
color output, 7Mhz.

Libertarian, atheist, semi-anarchal Techno-Rat.


En el caso de los 2 documentos que no coincide la clase, hay bastante similitud ya que las que dieron similitud de software y programación gráfica. En el caso del de menor similitud (4028) sí habla de tecnología y por eso la similitud, pero en el de mayor similitud (4347) aparentemente es la misma persona que escribe una frase al final del documento como una especie de firma (libertarian, anarchal, techno, rat, atheist y semi) y obviamente coinciden todas las palabras de esta frase.

In [135]:
print(newsgroups_train.data[4028])

Hello, and thank you for reading this request.  I have a Mpeg viewer for x-windows and it did not run because I was running it on a monochrome monitor.  I need the mono-driver for mpeg_play.   


In [136]:
print(newsgroups_train.data[4347])



It's really not that hard to do.  There are books out there which explain
everything, and the basic 3D functions, translation, rotation, shading, and
hidden line removal are pretty easy.  I wrote a program in a few weeks witht
he help of a book, and would be happy to give you my source.
	Also, Quickdraw has a lot of 3D functions built in, and Think pascal
can access them, and I would expect that THINK C could as well.  If you can
find out how to use the Quickdraw graphics library, it would be an excellent
choice, since it has a lot of stuff, and is built into the Mac, so should be
fast.

Libertarian, atheist, semi-anarchal Techno-Rat.


In [137]:
# cuarto documento de los 5 elegidos al azar
mostrar_similares(idxs[3], 4)

Caso 4  |  índice 4793  |  clase: talk.politics.guns


índice,clase,similitud,misma clase
6894,talk.politics.guns,0.236000,✓
5856,sci.crypt,0.236000,✗
4271,talk.politics.misc,0.233000,✗
3141,talk.politics.guns,0.230000,✓
10836,alt.atheism,0.229000,✗


En este caso vemos que hay 3 documentos que no coinciden la clase. En el caso del documento 4271, es muy similar el tópico de la clase ya que coincide talk.politics pero difiere de misc a guns.

In [138]:
#analicemos la coincidencia con el documento de mayor similitud que no coincidió la clase
terminos_compartidos(4793, 4271)


[('the', np.float64(0.045)),
 ('to', np.float64(0.033)),
 ('that', np.float64(0.024)),
 ('of', np.float64(0.023)),
 ('and', np.float64(0.019)),
 ('in', np.float64(0.012)),
 ('national', np.float64(0.009)),
 ('think', np.float64(0.009)),
 ('this', np.float64(0.006)),
 ('be', np.float64(0.005))]

In [139]:
terminos_compartidos(4793, 10836)

[('the', np.float64(0.058)),
 ('of', np.float64(0.028)),
 ('that', np.float64(0.025)),
 ('to', np.float64(0.021)),
 ('in', np.float64(0.014)),
 ('and', np.float64(0.009)),
 ('it', np.float64(0.007)),
 ('be', np.float64(0.007)),
 ('is', np.float64(0.007)),
 ('nature', np.float64(0.005))]

En ambos casos vemos que las palabras que más similitud aportan son stop_words

In [140]:
terminos_compartidos(4793, 5856)

[('the', np.float64(0.065)),
 ('of', np.float64(0.021)),
 ('to', np.float64(0.021)),
 ('that', np.float64(0.015)),
 ('and', np.float64(0.012)),
 ('in', np.float64(0.011)),
 ('be', np.float64(0.009)),
 ('this', np.float64(0.009)),
 ('it', np.float64(0.009)),
 ('protect', np.float64(0.007))]

En este caso tampoco hay coincidencia de ninguna palabra con sentido de lo que refiere el documento.

In [141]:
# quinto documento de los 5 elegidos al azar
mostrar_similares(idxs[4], 5)

Caso 5  |  índice 3813  |  clase: rec.sport.hockey


índice,clase,similitud,misma clase
10836,alt.atheism,0.251000,✗
759,soc.religion.christian,0.248000,✗
913,alt.atheism,0.241000,✗
5826,soc.religion.christian,0.241000,✗
5856,sci.crypt,0.233000,✗


En este caso vemos que en ninguno de los casos de mayor similitud coinciden la clase. Igualmente se observa que en todos los casos hay baja similitud.

In [142]:
#primero vamos a analizar el documento de referencia
print(newsgroups_train.data[3813])


Doesn't it also have the Statue of Liberty on it or is that Richter's Mask?

The back actually has a Bee followed by a Z to represent the Beezer. It 
also has something that looks like the three interconnecting circles from
the Led Zepplin 4 album cover. Is that what it is supposed to be? and if
it is does anybody know why he would put it there? Ali?


John
"The official Language of Golf is Profanity"





In [143]:
terminos_compartidos(3813, 10836)

[('the', np.float64(0.063)),
 ('is', np.float64(0.044)),
 ('it', np.float64(0.028)),
 ('that', np.float64(0.023)),
 ('of', np.float64(0.018)),
 ('to', np.float64(0.011)),
 ('also', np.float64(0.005)),
 ('be', np.float64(0.005)),
 ('has', np.float64(0.005)),
 ('there', np.float64(0.004))]

El mensaje de referencia habla de la máscara de un arquero  pero no usa ninguna palabra típica de hockey. Ya analizando el documento de referencia con el documento de mayor similitud se observa que hay muchas coincidencias de palabras que no tienen ningún significado en concreto. Estimo que en los casos de menor similitud es igual.

Documentos vacíos. Durante el análisis exploratorio se detectó que algunos documentos del conjunto de entrenamiento están vacíos. Esto ocurre porque el dataset se cargó con remove=('headers', 'footers', 'quotes'), y algunos mensajes solo contenían encabezados, firmas o citas de otros mensajes, por lo que al eliminarlos no queda texto. En estos casos la similitud es cero y el modelo da por defecto los últimos documentos.

> ### Conclusión
>
> La similitud coseno sobre vectores TF-IDF captura **coincidencias de vocabulario, no de significado**. En los 5 casos analizados se observaron los siguientes patrones:
>
> - **Similitud con sentido temático:** cuando los documentos comparten términos técnicos o específicos, la similitud refleja el tema, aunque la etiqueta no siempre coincida (caso 2, donde el término *dma* relaciona el documento con `ibm.pc.hardware`).
> - **Confusión entre clases cercanas:** muchos errores se dan entre grupos temáticamente próximos (`mac.hardware` / `ibm.pc.hardware`, `politics.guns` / `politics.misc`), donde el vocabulario se superpone naturalmente.
> - **Firmas y direcciones de email:** en los casos 1 y 3, la similitud más alta con otra clase se debe a la firma del autor o a su dirección de email, que se repiten en todos sus mensajes, independientemente del tema.
> - **Stopwords:** en los casos 4 y 5, casi toda la similitud proviene de palabras sin contenido (*the, of, to, that...*). Esto afecta especialmente a los **mensajes cortos**, donde hay pocos términos con contenido que compensen. El IDF no los logra anular ya que tiene el "+1", así que con eso conservan siempre un peso mínimo pero la suma de los mismos termina dominando.
>
> Como conclusión, es necesario un **preprocesamiento** del texto antes de medir similitud: eliminación de stopwords, y eventualmente de firmas, emails y términos muy frecuentes o muy raros.

### **2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**

In [144]:
# Clasificación por prototipos: cada documento de test toma la clase
# del documento de train más similar (1-NN con similitud coseno)
y_pred_proto = np.zeros(X_test.shape[0], dtype=int)
lote = 1000  # procesamos por partes para no ocupar demasiada memoria

for inicio in range(0, X_test.shape[0], lote):
    sim = cosine_similarity(X_test[inicio:inicio + lote], X_train)
    y_pred_proto[inicio:inicio + lote] = y_train[sim.argmax(axis=1)]

f1_proto = f1_score(y_test, y_pred_proto, average='macro')
print(f'F1-score macro (prototipos): {f1_proto:.4f}')


F1-score macro (prototipos): 0.5050


#### Conclusión:

- **Prototipos (zero-shot):** compara contra un solo documento, el más parecido. Si ese documento coincide por ruido (stopwords, firmas, emails), se equivoca.
- **Naive Bayes:** compara contra el resumen de todos los documentos de cada clase. El ruido de documentos individuales se diluye.

Esto le permite a Naive Bayes obtener un mejor F1-score macro (0.585 frente a 0.505). 
Con 20 clases, adivinar al azar da un F1 de alrededor de 0,05. Nuestro modelo da 0,505, unas 10 veces más. Eso demuestra que la similitud coseno sí captura información útil, aunque rinda menos que Naive Bayes


### **3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

Primero vamos a lematizar los documentos. Se tokeniza con el mismo regex  que usa TfidfVectorizer para que sea consistente.
lru_cache  va a guardar las palabras ya lematizadas, así las repetidas no se recalculan.

In [145]:
import re
import time
import itertools
import nltk
import pandas as pd
from functools import lru_cache
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

nltk.download("wordnet")
nltk.download("omw-1.4")

TOKEN_RE = re.compile(r"(?u)\b\w\w+\b")
lematizacion = WordNetLemmatizer()

@lru_cache(maxsize=None)
def lematizar(palabra):
    return lematizacion.lemmatize(palabra)

def lematizar_texto(texto):
    tokens = TOKEN_RE.findall(texto.lower())
    return " ".join(t if t in ENGLISH_STOP_WORDS else lematizar(t) for t in tokens)

[nltk_data] Downloading package wordnet to C:\Users\Gonzalo
[nltk_data]     Rodriguez\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Gonzalo
[nltk_data]     Rodriguez\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Aplicamos la lematización al test y train.
Imprimimos una muestra de ambos

In [146]:
t0 = time.time()
train_lem = [lematizar_texto(d) for d in newsgroups_train.data]
test_lem = [lematizar_texto(d) for d in newsgroups_test.data]
print(f"Lematización lista en {time.time() - t0:.0f} s")

print("\nOriginal:\n", newsgroups_train.data[0][:300])
print("\nLematizado:\n", train_lem[0][:300])

Lematización lista en 2 s

Original:
 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I k

Lematizado:
 was wondering if anyone out there could enlighten me on this car saw the other day it was door sport car looked to be from the late 60 early 70 it was called bricklin the door were really small in addition the front bumper was separate from the rest of the body this is all know if anyone can tellme 


Vamos a definir una grilla con los valores que se van a probar de la vectorización a través de TfidfVectorizer, los parámetros que vamos
a modificar son: stop_words (eliminar palabras que no aportan sentido), min_df (palabras raras que aparecen en muy pocos documentos), max_df (eliminar palabras que aparecen en muchos documentos), sublinear_tf (suaviza el peso de palabras muy repetidas) y alpha (suavizado de Laplace del modelo NB)

In [147]:
grid = {
    "stop_words": [None, "english"],
    "min_df": [1, 2, 5],
    "max_df": [0.8, 0.5],
    "sublinear_tf": [False, True],
}
alphas = [1.0, 0.1, 0.01]

n_combinaciones = len(list(itertools.product(*grid.values())))
print(f"{n_combinaciones} configuraciones de vectorizador × 2 modelos × {len(alphas)} alphas "
      f"= {n_combinaciones * 2 * len(alphas)} experimentos")

24 configuraciones de vectorizador × 2 modelos × 3 alphas = 144 experimentos


Se ajusta la vectorización solo con train (fit_transform) y en cada test (transform) para que el test no cambie de valores.
Vamos a entrenar MultinomialNB y ComplementNB con cada alpha.
Se guardan los F1 sobre test y el tamaño del vocabulario

In [148]:
resultados = []
t0 = time.time()

for valores in itertools.product(*grid.values()):
    params = dict(zip(grid.keys(), valores))
    vec = TfidfVectorizer(**params)
    X_tr = vec.fit_transform(train_lem)
    X_te = vec.transform(test_lem)
    for Modelo in (MultinomialNB, ComplementNB):
        for a in alphas:
            y_pred = Modelo(alpha=a).fit(X_tr, y_train).predict(X_te)
            f1 = f1_score(y_test, y_pred, average="macro")
            resultados.append({**params, "modelo": Modelo.__name__,
                               "alpha": a, "f1": f1, "vocab": X_tr.shape[1]})

print(f"Grilla terminada en {(time.time() - t0) / 60:.1f} min")

Grilla terminada en 0.7 min


Observemos los resultados obtenidos

In [149]:
df = pd.DataFrame(resultados).sort_values("f1", ascending=False).reset_index(drop=True)
display(df.head(10))

mejor = df.iloc[0]
params_mejor = {k: (None if pd.isna(mejor[k]) else mejor[k]) for k in grid}
print("Mejor configuración:", params_mejor)
print(f"Modelo: {mejor['modelo']}, alpha={mejor['alpha']}")
print(f"F1 macro: {mejor['f1']:.4f} | Vocabulario: {mejor['vocab']} palabras")

,stop_words,min_df,max_df,sublinear_tf,modelo,alpha,f1,vocab
0,NaN,2,0.8,True,ComplementNB,1.0,0.691105,35892
1,NaN,2,0.5,True,ComplementNB,1.0,0.690956,35884
2,english,2,0.5,False,ComplementNB,1.0,0.690793,35585
3,english,2,0.8,False,ComplementNB,1.0,0.690793,35585
4,NaN,2,0.5,False,ComplementNB,1.0,0.690740,35884
5,english,2,0.8,True,ComplementNB,1.0,0.690533,35585
6,english,2,0.5,True,ComplementNB,1.0,0.690533,35585
7,NaN,2,0.8,False,ComplementNB,1.0,0.690265,35892
8,NaN,1,0.5,False,ComplementNB,1.0,0.689975,95924
9,english,1,0.8,False,ComplementNB,1.0,0.689958,95624


Mejor configuración: {'stop_words': None, 'min_df': np.int64(2), 'max_df': np.float64(0.8), 'sublinear_tf': np.True_}
Modelo: ComplementNB, alpha=1.0
F1 macro: 0.6911 | Vocabulario: 35892 palabras


#### Conclusión

La mejor configuración obtuvo un **F1 macro de 0,6911**, frente al **0,5854** del baseline de la plantilla (vectorizador con parámetros por defecto + MultinomialNB). La mejora combina dos cambios: ajuste de los parámetros del vectorizador y el uso de ComplementNB.

**Mejor configuración:** ComplementNB con `alpha=1.0`, `min_df=2`, `max_df=0.8`, `sublinear_tf=True` y sin `stop_words`, con un vocabulario de 35.892 palabras.

- **ComplementNB:** observamos que los mejores modelos que performan son todos del mismo tipo, ya que utiliza los complementos para estimar y las probabilidades son más estables.
- **min_df:** comparando los resultados entre la línea 0 y la 8 se ve que el vocabulario baja de 95.924 a 35.892 palabras e incluso el F1 sube un poco. Esto indica que había aprox. un 63% de palabras que aparecían en un solo documento. Se disminuyó mucho la dimensión y el rendimiento es casi similar.
- **max_df:** vemos que en los 2 valores que se probaron (0.8 y 0.5) solo hay 8 palabras de diferencia; casi no hay palabras que aparezcan en más de la mitad de los documentos.
- **stop_words (english):** vemos que con este parámetro activado, max_df (0.8 y 0.5) da el mismo vocabulario (35.585), indicando que esas pocas palabras muy frecuentes eran todas stop_words. Vemos que stop_words eliminó unas 300 palabras sin mejorar el F1, ya que el IDF las estaba aplastando porque aparecían en muchos documentos.
- **alpha:** los mejores modelos usan el valor 1, que es el valor más alto de los que se probaron, por lo que es el mejor alpha dentro de la grilla evaluada.

Por último vamos a correr el modelo entrenado pero con los valores sin lematizar para ver si mejora o no el F1

In [150]:
Modelo = MultinomialNB if mejor["modelo"] == "MultinomialNB" else ComplementNB

vec = TfidfVectorizer(**params_mejor)
X_tr = vec.fit_transform(newsgroups_train.data)
X_te = vec.transform(newsgroups_test.data)
y_pred = Modelo(alpha=mejor["alpha"]).fit(X_tr, y_train).predict(X_te)
f1_sin_lem = f1_score(y_test, y_pred, average="macro")

pd.DataFrame({
    "F1 macro": [mejor["f1"], f1_sin_lem],
    "Vocabulario": [mejor["vocab"], X_tr.shape[1]],
}, index=["Con lematización", "Sin lematización"])

,F1 macro,Vocabulario
Con lematización,0.691105,35892
Sin lematización,0.693445,39422


**Lematización:** con los mismos parámetros, sin lematizar se obtiene un F1 de 0.6934 con un vocabulario de 39.422 palabras, contra 

0.6911 y 35.892 palabras lematizando. La lematización redujo el vocabulario un 9% pero no mejoró el F1, la diferencia es muy chica. Esto puede ser porque el lematizador sin etiqueta gramatical solo unifica plurales y a veces junta palabras que convenía separar, por ejemplo "windows" (sistema operativo) pasa a "window".

### **4. Transponer la matriz documento-término.**

In [151]:
# Transponemos: cada fila pasa a ser un término representado por sus pesos en cada documento
#X_terminos = X_train.T.tocsr()
#print(f'shape: {X_terminos.shape}  (términos × documentos)')

#def palabras_similares(palabra, n=5):
    #idx = tfidfvect.vocabulary_[palabra]
    #sim = cosine_similarity(X_terminos[idx], X_terminos)[0]
    #mas_sim = np.argsort(sim)[::-1][1:n + 1]  # excluimos la propia palabra
    #print(f"Palabra: '{palabra}'  |  aparece en {X_terminos[idx].nnz} documentos")
    #tabla = pd.DataFrame({
    #    'palabra similar': [idx2word[i] for i in mas_sim],
    #    'similitud': sim[mas_sim].round(3),
    #})
    #display(tabla.style.hide(axis='index'))


Vamos a hacer una función que me seleccione palabras al azar y yo voy a ir armando a mano una lista con las palabras que más sirvan para el problema planteado.

In [152]:
# Cantidad de documentos en los que aparece cada palabra del vocabulario
docs_por_palabra = np.asarray((X_train > 0).sum(axis=0)).ravel()

def palabras_al_azar(n=10, min_docs=50):
    # Filtramos palabras alfabéticas que aparezcan en al menos min_docs documentos,
    # para evitar números, códigos y términos muy raros
    candidatas = [i for i in range(len(idx2word))
                  if idx2word[i].isalpha() and docs_por_palabra[i] >= min_docs]
    elegidas = np.random.choice(candidatas, size=n, replace=False)
    for i in elegidas:
        print(f'{idx2word[i]:<15} aparece en {docs_por_palabra[i]} documentos')


In [153]:
palabras_al_azar()


bodies          aparece en 60 documentos
share           aparece en 147 documentos
fill            aparece en 64 documentos
images          aparece en 112 documentos
quickly         aparece en 134 documentos
term            aparece en 200 documentos
defense         aparece en 191 documentos
fault           aparece en 95 documentos
troops          aparece en 74 documentos
release         aparece en 180 documentos


In [154]:
palabras = ['gun', 'bike', 'disk', 'crimes', 'windows']

Vamos a definir una función para encontrar la similitud con las 5 palabras seleccionadas en la lista

In [155]:
def palabras_similares(palabra, n_caso):
    idx = tfidfvect.vocabulary_[palabra]
    sim = cosine_similarity(X_terminos[idx], X_terminos)[0]
    mas_sim = np.argsort(sim)[::-1][1:6]  # las 5 más similares, excluyendo la propia palabra

    print(f"Caso {n_caso}  |  palabra: '{palabra}'  |  aparece en {X_terminos[idx].nnz} documentos")

    tabla = pd.DataFrame({
        'palabra similar': [idx2word[i] for i in mas_sim],
        'similitud': sim[mas_sim].round(3),
        'documentos': [X_terminos[i].nnz for i in mas_sim],
    })
    display(tabla.style.hide(axis='index'))


def documentos_compartidos(p1, p2, n=5):
    # Documentos donde aparecen ambas palabras y cuánto aporta cada uno a su similitud
    a = X_terminos[tfidfvect.vocabulary_[p1]].toarray()[0]
    b = X_terminos[tfidfvect.vocabulary_[p2]].toarray()[0]
    aporte = a * b
    top = [d for d in aporte.argsort()[::-1][:n] if aporte[d] > 0]

    return pd.DataFrame({
        'documento': top,
        'clase': [nombres[y_train[d]] for d in top],
        'aporte': aporte[top].round(3),
    })


In [ ]:
# primera palabra
palabras_similares(palabras[0], 1)

Caso 1  |  palabra: 'gun'  |  aparece en 235 documentos


palabra similar,similitud,documentos
guns,0.358000,156
crime,0.244000,172
handgun,0.239000,41
homicides,0.233000,15
firearms,0.233000,76


En esta primera palabra gun (arma), se observa mucha coherencia en las palabras seleccionadas por similitud, ya que se observan palabras como gun (arma), crime (crimen), handgun (arma de mano), homicides (homicidios) y firearms (armas de fuego). Además se agrego una columna con la cantidad de documentos donde aparece esta palabra.

In [157]:
# segunda palabra
palabras_similares(palabras[1], 2)

Caso 2  |  palabra: 'bike'  |  aparece en 183 documentos


palabra similar,similitud,documentos
bikes,0.277000,65
ride,0.272000,99
cbr,0.238000,11
steet,0.226000,1
gsxr,0.212000,2


En el caso de la segunda palabra seleccionada, bike, vamos a ver si se refiere a la jerga de bicicletas o motocicletas

In [158]:
documentos_compartidos('bike', 'ride')

,documento,clase,aporte
0,6323,rec.motorcycles,0.118
1,8771,rec.motorcycles,0.057
2,2951,rec.motorcycles,0.056
3,3067,rec.motorcycles,0.049
4,9320,rec.motorcycles,0.044


Evidentemente se refiere a motocicleta entonces tenemos mucha coherencia en varias palabras. Empezando por las de mayor similitud y siguiendo por CBR y GSXR que son modelos específicos de Honda y Suzuki. El último punto a analizar es steet que se supone que es un error de tipeo en street y por eso está en un solo documento y como es única realza su peso en el único documento donde está.

In [159]:
# tercera palabra
palabras_similares(palabras[2], 3)

Caso 3  |  palabra: 'disk'  |  aparece en 275 documentos


palabra similar,similitud,documentos
hard,0.261000,576
floppy,0.256000,126
drive,0.251000,490
boot,0.211000,72
hinds,0.204000,2


En esta tercera palabra disk se observa mucha coherencia en las palabras con mayor similitud, solo no se encuentra relacionamiento con el último caso hinds.

In [160]:
documentos_compartidos('disk', 'hinds')

,documento,clase,aporte
0,10834,comp.sys.ibm.pc.hardware,0.112
1,750,sci.med,0.000


In [161]:
#primero vamos a analizar el documento de referencia
print(newsgroups_train.data[10834])

Does anyone have enough experience to report whether disk data
compression has any effect on the optimal disk sector interleave?
Offhand, I expect that the time required to decompress disk data
would increase the optimum disk sector interleave.
                                              - Alan Hinds



In [162]:
#primero vamos a analizar el documento de referencia
print(newsgroups_train.data[750])

------------- cut here -----------------
Volume  6, Number 10                                           April 20, 1993

              +------------------------------------------------+
              !                                                !
              !              Health Info-Com Network           !
              !                Medical Newsletter              !
              +------------------------------------------------+
                         Editor: David Dodell, D.M.D.
    10250 North 92nd Street, Suite 210, Scottsdale, Arizona 85258-4599 USA
                          Telephone +1 (602) 860-1121
                              FAX +1 (602) 451-1165

Compilation Copyright 1993 by David Dodell,  D.M.D.  All  rights  Reserved.  
License  is  hereby  granted  to republish on electronic media for which no 
fees are charged,  so long as the text of this copyright notice and license 
are attached intact to any and all republished portion or portions.  

The Health Info-

En estos casos donde muestra similitud de disk con hinds, es una coincidencia ya que en ambos documentos hay personas mencionadas con apellidos hinds. Se observa que hay solo dos documentos donde coinciden, dicho número es mucho menor que el resto de las palabras de mayor similitud.

In [163]:
# cuarta palabra
palabras_similares(palabras[3], 4)

Caso 4  |  palabra: 'crimes'  |  aparece en 66 documentos


palabra similar,similitud,documentos
jepardy,0.436000,1
differnt,0.436000,1
blacks,0.379000,27
sovereign,0.366000,6
bombers,0.319000,8


En principio lo que vemos es que la palabra aparece en muy pocos documentos (66). Vemos que los 2 primeros valores de mayor similitud tienen el mismo valor y presuponemos que son errores de tipeo.

In [164]:
#vamos a analizar el documento de referencia
documentos_compartidos('crimes', 'jepardy')

,documento,clase,aporte
0,2068,talk.politics.misc,0.051


In [165]:
print(newsgroups_train.data[2068])




But you still get into trouble. For example - how often are crimes
of violence not "hate crimes"? The question is then who are you
hating? If its another gang member, then its better than if
the person you hate is of a differnt color? 

Also, is it realistic to declare that crimes of hate are worse
than crimes of gross negligence? (Like random drive by shootings
where they can't be hate crimes because the shooter doesn't know
who he is going to hit - he just shoots into the crowd).



First, I heard today that there is a good chance that the U.S. instead
of New York is going after the bombers. This means no capital punishment.

Secondly, double jepardy does help keep the government from going after
you for first one murder, then the next, etc. A "sovereign" has essentially
one chance with a single fact pattern (such as the World Trade Center bombing).
That is why the bombers will in all probability be tried for all the
deaths, as well as the property damages, as well as conspiracy, 

Vemos que las 2 palabras con mayor similitud aparecen en el mismo documento y además es el único documento donde aparecen, por el hecho de estar mal tipeadas. Las palabras correctas deberían ser jeopardy (peligro) y different (diferente).

In [166]:
#Analizamos los documentos que relacionan ambas palabras
documentos_compartidos('crimes', 'blacks')

,documento,clase,aporte
0,7703,talk.politics.misc,0.147
1,10011,talk.politics.misc,0.143
2,8031,talk.politics.misc,0.011
3,10986,talk.politics.misc,0.011
4,8862,talk.politics.misc,0.007


Se ve que la temática de los documentos está relacionada con temas políticos que relacionan crímenes y raza. Además observamos que los 2 primeros documentos son los que mayor aporte tienen a la totalidad del 0.379 de similitud.

Además el documento 2068 se comparten las palabras de similitud (jepardy, differnt, sovereign y bombers) que discute crímenes de odio, atentado  y el principio del doble jeopardy. Por lo que concluimos que para  palabras poco frecuentes como  'crimes' un solo documento puede definir casi todos sus vecinos.

In [167]:
# quinta palabra
palabras_similares(palabras[4], 5)

Caso 5  |  palabra: 'windows'  |  aparece en 575 documentos


palabra similar,similitud,documentos
dos,0.304000,281
ms,0.232000,177
microsoft,0.222000,98
nt,0.214000,79
for,0.193000,6294


En este caso de la palabra Windows hay una relación directa con las primeras cuatro palabras de similitud, son sistemas operativos relacionados (MS-DOS, WINDOWS NT) y Microsoft es el propietario. En el caso de for que es un stopword, aunque su peso TF-IDF en cada documento es bajo, pero a pesar de eso seguramente comparte muchos documentos de los 575 donde esta windows por lo que la suma de todos estos pequeños valores lo termina metiendo en el quinto lugar de similitud.

Los temas más importantes a resaltar son que se encontraron 2 tipos de "similitud falsa":
- **crimes:** un solo documento con mucho peso puede definir la similitud
- **for:** muchísimos documentos con poco peso cada uno suman a la similitud

Vemos que estos vectores de palabras capturan co-ocurrencia (palabras que aparecen en el mismo documento), no significado.  